In [2]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import joblib

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Joblib:", joblib.__version__)
print("Environment check completed!")

Python: 3.12.7
NumPy: 2.5.4
Pandas: 2.2.3
Scikit-learn: 1.9.1
Matplotlib: 3.10.1
Joblib: 1.4.2
Environment check completed!


In [4]:
%pip install scikit-learn matplotlib joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 158.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 145.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load the input features and target labels
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()

# Encode malignant as 1 and benign as 0
y = (dataset.target == 0).astype(int)
y.name = "diagnosis"

# Combine features and target for inspection
df = X.copy()
df["diagnosis"] = y

print("Number of records:", X.shape[0])
print("Number of input features:", X.shape[1])
print("Missing values:", df.isna().sum().sum())
print("\nClass counts (0 = benign, 1 = malignant):")
print(df["diagnosis"].value_counts())

display(df.head())

Number of records: 569
Number of input features: 30
Missing values: 0

Class counts (0 = benign, 1 = malignant):
diagnosis
0    357
1    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,1
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Test records:", X_test.shape[0])
print("Input features:", X_train.shape[1])

# Save unseen test records for the Streamlit app
test_data = X_test.copy()
test_data["diagnosis"] = y_test
test_data.to_csv("test_data.csv", index=False)

print("\nTest data saved as test_data.csv")
print("Test class counts (0 = benign, 1 = malignant):")
print(y_test.value_counts())

Training records: 455
Test records: 114
Input features: 30

Test data saved as test_data.csv
Test class counts (0 = benign, 1 = malignant):
diagnosis
0    72
1    42
Name: count, dtype: int64


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# Fit scaling and Logistic Regression on training data only
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

logistic_model.fit(X_train, y_train)

# Predict on unseen test data
y_pred = logistic_model.predict(X_test)
y_probability = logistic_model.predict_proba(X_test)[:, 1]

logistic_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_probability),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred)
}

results = [logistic_results]
display(pd.DataFrame(results).round(4))

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9649,0.996,0.975,0.9286,0.9512,0.9245


In [12]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

tree_predictions = decision_tree_model.predict(X_test)
tree_probabilities = decision_tree_model.predict_proba(X_test)[:, 1]

tree_results = {
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, tree_predictions),
    "AUC": roc_auc_score(y_test, tree_probabilities),
    "Precision": precision_score(
        y_test, tree_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, tree_predictions, zero_division=0
    ),
    "F1": f1_score(y_test, tree_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, tree_predictions)
}

# Replace any previous Decision Tree result if this cell is rerun
results = [row for row in results if row["Model"] != "Decision Tree"]
results.append(tree_results)

display(pd.DataFrame(results).round(4))

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492


In [14]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_train, y_train)

knn_predictions = knn_model.predict(X_test)
knn_probabilities = knn_model.predict_proba(X_test)[:, 1]

knn_results = {
    "Model": "KNN",
    "Accuracy": accuracy_score(y_test, knn_predictions),
    "AUC": roc_auc_score(y_test, knn_probabilities),
    "Precision": precision_score(
        y_test, knn_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, knn_predictions, zero_division=0
    ),
    "F1": f1_score(y_test, knn_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, knn_predictions)
}

# Prevent duplicate rows if this cell is rerun
results = [row for row in results if row["Model"] != "KNN"]
results.append(knn_results)

display(pd.DataFrame(results).round(4))

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
2,KNN,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058


In [16]:
from sklearn.naive_bayes import GaussianNB

naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train, y_train)

nb_predictions = naive_bayes_model.predict(X_test)
nb_probabilities = naive_bayes_model.predict_proba(X_test)[:, 1]

nb_results = {
    "Model": "Naive Bayes",
    "Accuracy": accuracy_score(y_test, nb_predictions),
    "AUC": roc_auc_score(y_test, nb_probabilities),
    "Precision": precision_score(
        y_test, nb_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, nb_predictions, zero_division=0
    ),
    "F1": f1_score(y_test, nb_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, nb_predictions)
}

# Prevent duplicate rows if this cell is rerun
results = [
    row for row in results
    if row["Model"] != "Naive Bayes"
]
results.append(nb_results)

display(pd.DataFrame(results).round(4))

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
2,KNN,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058
3,Naive Bayes,0.9386,0.9934,1.0000,0.8333,0.9091,0.8715


In [18]:
from sklearn.svm import SVC

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        probability=True,
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

svm_predictions = svm_model.predict(X_test)
svm_probabilities = svm_model.predict_proba(X_test)[:, 1]

svm_results = {
    "Model": "Support Vector Machine",
    "Accuracy": accuracy_score(y_test, svm_predictions),
    "AUC": roc_auc_score(y_test, svm_probabilities),
    "Precision": precision_score(
        y_test, svm_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, svm_predictions, zero_division=0
    ),
    "F1": f1_score(y_test, svm_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, svm_predictions)
}

# Prevent duplicate rows if this cell is rerun
results = [
    row for row in results
    if row["Model"] != "Support Vector Machine"
]
results.append(svm_results)

comparison_df = pd.DataFrame(results)
display(comparison_df.round(4))

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
2,KNN,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058
3,Naive Bayes,0.9386,0.9934,1.0000,0.8333,0.9091,0.8715
4,Support Vector Machine,0.9737,0.9947,1.0000,0.9286,0.9630,0.9442


In [20]:
from pathlib import Path
import joblib
import json
import sklearn
import numpy as np
import pandas as pd

model_folder = Path("model")
model_folder.mkdir(exist_ok=True)

trained_models = {
    "logistic_regression": logistic_model,
    "decision_tree": decision_tree_model,
    "knn": knn_model,
    "naive_bayes": naive_bayes_model,
    "svm": svm_model
}

for name, model in trained_models.items():
    file_path = model_folder / f"{name}.joblib"
    joblib.dump(model, file_path)

    # Verify that saving and loading preserves predictions
    loaded_model = joblib.load(file_path)
    assert np.array_equal(
        model.predict(X_test),
        loaded_model.predict(X_test)
    )
    print(f"Saved and verified: {file_path}")

metadata = {
    "feature_names": X_train.columns.tolist(),
    "target_column": "diagnosis",
    "class_labels": {"0": "benign", "1": "malignant"},
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "joblib_version": joblib.__version__
}

with open(model_folder / "metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

comparison_df = pd.DataFrame(results)
comparison_df.to_csv("model_comparison.csv", index=False)

print("\nMetadata and comparison table saved.")
print("Project location:", Path.cwd())

Saved and verified: model/logistic_regression.joblib
Saved and verified: model/decision_tree.joblib
Saved and verified: model/knn.joblib
Saved and verified: model/naive_bayes.joblib
Saved and verified: model/svm.joblib

Metadata and comparison table saved.
Project location: /home/s2026ak05557/ML -Assignment -2


In [22]:
%pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 137.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 133.4 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.

In [24]:
import streamlit as st

print("Streamlit version:", st.__version__)
print("Ready to create the app!")

Streamlit version: 1.66.1
Ready to create the app!


In [26]:
%%writefile app.py
from pathlib import Path
import json
import streamlit as st

st.set_page_config(
    page_title="Breast Cancer Model Explorer",
    page_icon="📊",
    layout="wide"
)

BASE_DIR = Path(__file__).resolve().parent

with open(BASE_DIR / "model" / "metadata.json") as file:
    metadata = json.load(file)

MODEL_FILES = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}

st.title("Breast Cancer Model Explorer")
st.write(
    "Compare five classification models using "
    "cell-nucleus measurements from the Wisconsin dataset."
)
st.caption("Machine Learning Assignment 2 | Educational demonstration")

selected_model = st.sidebar.selectbox(
    "Choose a classification model",
    list(MODEL_FILES)
)

st.sidebar.info("Target labels: 0 = benign, 1 = malignant")

st.subheader("Upload your test data")
st.write(
    "Upload a CSV containing the 30 input features and "
    "the diagnosis column."
)

uploaded_file = st.file_uploader(
    "Choose a test CSV",
    type=["csv"]
)

sample_path = BASE_DIR / "test_data.csv"
if sample_path.exists():
    st.download_button(
        "Download sample test CSV",
        data=sample_path.read_bytes(),
        file_name="test_data.csv",
        mime="text/csv"
    )

if uploaded_file is None:
    st.info("Upload test data to begin evaluating a model.")
else:
    st.success("CSV uploaded. Model evaluation will be added next.")

Writing app.py


In [28]:
%%writefile -a app.py

# Continue only after a CSV is uploaded
if uploaded_file is not None:
    import pandas as pd
    import numpy as np
    import joblib
    from sklearn.metrics import (
        accuracy_score, roc_auc_score, precision_score,
        recall_score, f1_score, matthews_corrcoef,
        confusion_matrix, classification_report
    )

    try:
        test_df = pd.read_csv(uploaded_file)
    except Exception as error:
        st.error(f"Could not read the CSV: {error}")
        st.stop()

    features = metadata["feature_names"]
    target = metadata["target_column"]
    required_columns = features + [target]

    missing = [
        column for column in required_columns
        if column not in test_df.columns
    ]

    if missing:
        st.error(f"Missing required columns: {', '.join(missing)}")
        st.stop()

    if test_df.empty:
        st.error("The CSV has no data rows.")
        st.stop()

    try:
        inputs = test_df[features].apply(
            pd.to_numeric, errors="raise"
        )
        actual = pd.to_numeric(test_df[target], errors="raise")
    except (ValueError, TypeError):
        st.error("All features and diagnosis must be numeric.")
        st.stop()

    if not np.isfinite(inputs.to_numpy(dtype=float)).all():
        st.error("Features must not contain missing or infinite values.")
        st.stop()

    if not actual.isin([0, 1]).all():
        st.error("Diagnosis must contain only 0 (benign) or 1 (malignant).")
        st.stop()

    actual = actual.astype(int)

    st.write(f"Test records: **{len(test_df)}**")
    with st.expander("Preview uploaded test data"):
        st.dataframe(test_df.head())

    @st.cache_resource
    def load_saved_model(filename):
        return joblib.load(BASE_DIR / "model" / filename)

    def evaluate_model(model):
        predictions = model.predict(inputs)
        positive_index = list(model.classes_).index(1)
        probabilities = model.predict_proba(inputs)[:, positive_index]

        scores = {
            "Accuracy": accuracy_score(actual, predictions),
            "AUC": (
                roc_auc_score(actual, probabilities)
                if actual.nunique() == 2 else np.nan
            ),
            "Precision": precision_score(
                actual, predictions, zero_division=0
            ),
            "Recall": recall_score(
                actual, predictions, zero_division=0
            ),
            "F1": f1_score(actual, predictions, zero_division=0),
            "MCC": matthews_corrcoef(actual, predictions)
        }
        return scores, predictions

    model = load_saved_model(MODEL_FILES[selected_model])
    scores, predictions = evaluate_model(model)

    st.subheader(f"Results: {selected_model}")
    metric_columns = st.columns(3)

    for index, (name, value) in enumerate(scores.items()):
        metric_columns[index % 3].metric(
            name,
            "N/A" if pd.isna(value) else f"{value:.4f}"
        )

    if actual.nunique() < 2:
        st.info("AUC requires both classes in the uploaded test data.")

    st.caption("Precision, recall and F1 use malignant (1) as positive.")

    st.subheader("Confusion matrix")
    matrix = confusion_matrix(actual, predictions, labels=[0, 1])
    st.dataframe(pd.DataFrame(
        matrix,
        index=["Actual benign", "Actual malignant"],
        columns=["Predicted benign", "Predicted malignant"]
    ))

    st.subheader("Classification report")
    report = classification_report(
        actual, predictions,
        labels=[0, 1],
        target_names=["Benign", "Malignant"],
        output_dict=True,
        zero_division=0
    )
    st.dataframe(pd.DataFrame(report).transpose().round(4))

    st.subheader("Compare all five models")
    comparison_rows = []

    for name, filename in MODEL_FILES.items():
        saved_model = load_saved_model(filename)
        model_scores, _ = evaluate_model(saved_model)
        comparison_rows.append({"Model": name, **model_scores})

    st.dataframe(pd.DataFrame(comparison_rows).round(4))

Appending to app.py


In [30]:
import subprocess
import sys
from pathlib import Path

# Check the app's Python syntax
app_path = Path("app.py").resolve()
compile(app_path.read_text(), str(app_path), "exec")

# Start Streamlit without blocking the notebook
log_file = open("streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(app_path),
        "--server.address=127.0.0.1",
        "--server.port=8501",
        "--server.headless=true",
        "--browser.gatherUsageStats=false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Open http://localhost:8501 in Firefox inside BITS Lab.")

Open http://localhost:8501 in Firefox inside BITS Lab.


In [32]:
from importlib.metadata import version
from pathlib import Path

packages = [
    "streamlit",
    "scikit-learn",
    "numpy",
    "pandas",
    "scipy",
    "joblib"
]

requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
) + "\n"

Path("requirements.txt").write_text(requirements)

print("requirements.txt created:\n")
print(requirements)

requirements.txt created:

streamlit==1.66.1
scikit-learn==1.9.1
numpy==2.5.4
pandas==2.2.3
scipy==1.18.1
joblib==1.4.2



In [34]:
from pathlib import Path
import pandas as pd

# Generate the metrics table from your saved results
metrics = pd.read_csv("model_comparison.csv")
metrics_table = (
    "| " + " | ".join(metrics.columns) + " |\n"
    + "| " + " | ".join(["---"] * len(metrics.columns)) + " |\n"
)

for _, row in metrics.iterrows():
    values = [
        str(row["Model"]),
        *[f"{row[column]:.4f}" for column in metrics.columns[1:]]
    ]
    metrics_table += "| " + " | ".join(values) + " |\n"

readme = f"""# Breast Cancer Model Explorer

## a. Problem statement
Compare five classification algorithms for classifying breast mass
samples as benign or malignant using numerical cell-nucleus measurements.
Provide an interactive Streamlit application to evaluate saved models
on uploaded test data.

## b. Dataset description
Dataset: UCI Breast Cancer Wisconsin (Diagnostic).
Source: https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic

The dataset contains 569 records and 30 numerical input features.
The features describe cell-nucleus characteristics, including radius,
texture, perimeter and area.

Target column: diagnosis.
Encoding used in this project: 0 = benign, 1 = malignant.
Class distribution: 357 benign and 212 malignant samples.

A stratified 80:20 split produces 455 training records and 114 test records,
using random_state=42. The test set contains 72 benign and 42 malignant
samples. StandardScaler is fitted only on training data within the
Logistic Regression, KNN and SVM pipelines.

## c. GitHub Repository Link
TO BE ADDED AFTER CREATING THE REPOSITORY

## d. Models used

- Logistic Regression: StandardScaler and max_iter=2000.
- Decision Tree: default settings with random_state=42.
- KNN: StandardScaler and five nearest neighbors.
- Gaussian Naive Bayes: default settings.
- SVM: StandardScaler, RBF kernel, C=1.0, probability=True,
  random_state=42.

### Evaluation metrics
All models are evaluated on the same held-out test set.
Precision, recall and F1 use malignant (1) as the positive class.
AUC uses predicted malignant-class probabilities.

{metrics_table}

### Observations

| ML Model Name | Observation about model performance |
| --- | --- |
| Logistic Regression | Accuracy 96.49%, with the highest AUC of 0.9960. It missed 3 malignant samples and incorrectly classified 1 benign sample as malignant. |
| Decision Tree | Accuracy 92.98%, the lowest among these five models on this split. It produced 4 false negatives and 4 false positives. |
| KNN | Accuracy 95.61%. It produced 4 false negatives and 1 false positive. |
| Naive Bayes | Precision 1.0000, with no false positives, but recall 0.8333. It missed 7 malignant samples. |
| Support Vector Machine | Highest accuracy of 97.37%, F1 of 0.9630 and MCC of 0.9442. It missed 3 malignant samples and produced no false positives. |
| Overall winner for this test split | SVM, based on accuracy, F1 and MCC. Logistic Regression achieved the highest AUC. |

These observations describe one held-out split and do not establish
performance on other populations. No hyperparameter search was performed.

## Running the application
Use Python 3.12 and install the dependencies:

    pip install -r requirements.txt

Launch the app:

    streamlit run app.py

Upload test_data.csv and select a model from the sidebar.
The CSV must contain all 30 original feature columns and diagnosis.
The app loads saved models without retraining and displays six metrics,
a confusion matrix, a classification report and a comparison of all models.
AUC is shown as N/A when the uploaded data contains only one class.

## Project files
- app.py: Streamlit application.
- requirements.txt: dependency versions.
- README.md: project description and findings.
- test_data.csv: held-out test records with target labels.
- model/: five saved models and metadata.json.
- model_comparison.csv: evaluation results.
- Training notebook (.ipynb): source code and execution outputs.

## Live Streamlit App Link
TO BE ADDED AFTER DEPLOYMENT

## Execution environment
Training and local application execution were performed on BITS Virtual Lab.
"""

Path("README.md").write_text(readme, encoding="utf-8")
print("README.md created successfully.")

README.md created successfully.
